In [2]:
class PortfolioChoiceEconomy:
    def __init__(self, YD_0, V_0, B_h_0, r_0=0.0):
        #Households can't hold more bills than their total wealth.
        if V_0 < B_h_0:
            raise ValueError("B_h_0 cannot be greater than V_0.")

        #B_cb_0 is residual: whatever wealth isn't held as bills starts out as central-bank money.
        B_cb_0 = V_0 - B_h_0
        self.period = [0]
        self.G = [0.0]
        self.Y = [0.0]
        self.T = [0.0]
        self.YD = [float(YD_0)]
        self.YD_e = [float(YD_0)]
        self.C = [0.0]
        self.V = [float(V_0)]
        self.V_e = [float(V_0)]
        self.B_d = [float(B_h_0)]
        self.H_d = [float(B_cb_0)]
        self.B_h = [float(B_h_0)]
        self.B_s = [float(V_0)]
        self.B_cb = [float(B_cb_0)]
        self.H_s = [float(B_cb_0)]
        self.H_h = [float(B_cb_0)]
        self.delta_Bs = [0.0]
        self.delta_Hs = [0.0]
        self.r = [float(r_0)]
        self.alpha1 = [0.0]

    #Set interest_sensitivity to zero for PCEX1; for PCEX2, pass the eq.
    #(4.31) parameter so alpha1_t = alpha10 - l*r(-1).
    def run_period(self, G, alpha1, alpha2, theta, r, lambda20, lambda21,
                   lambda22, interest_sensitivity=0.0):
        YD_e = self.YD[-1]                 #PCEX1, equation (4.16A)
        r_1 = self.r[-1]
        V_1 = self.V[-1]
        B_h_1 = self.B_h[-1]
        B_s_1 = self.B_s[-1]
        B_cb_1 = self.B_cb[-1]
        alpha1_t = alpha1 - interest_sensitivity * r_1  #PCEX2 when l > 0

        #Consumption and income
        C = alpha1_t * YD_e + alpha2 * V_1
        Y = C + G

        #Taxes and disposable income
        interest_to_households = r_1 * B_h_1
        T = theta * (Y + interest_to_households)
        YD = Y - T + interest_to_households

        #Wealth
        V = V_1 + (YD - C)
        V_e = V_1 + (YD_e - C)

        #Portfolio allocation
        if V_1 == 0:
            raise ZeroDivisionError("V from the previous period cannot be zero.")
        bill_share = lambda20 + lambda21 * r_1 - lambda22 * (YD_e / V_1)
        if not 0 <= bill_share <= 1:
            raise ValueError(
                f"Bill share is {bill_share:.3f}; choose parameters that keep it between 0 and 1."
            )
        B_d = V_e * bill_share
        H_d = V_e - B_d

        #Asset-market equilibrium and government/central-bank accounting.
        B_h = B_d
        delta_Bs = (G + r_1 * B_s_1) - (T + r_1 * B_cb_1)
        B_s = B_s_1 + delta_Bs
        B_cb = B_s - B_h
        delta_Hs = B_cb - B_cb_1
        H_s = self.H_s[-1] + delta_Hs
        H_h = H_s                           #redundant equation (4.12)

        self.period.append(self.period[-1] + 1)
        for name, value in {
            "G": G, "Y": Y, "T": T, "YD": YD, "YD_e": YD_e, "C": C,
            "V": V, "V_e": V_e, "B_d": B_d, "H_d": H_d, "B_h": B_h,
            "B_s": B_s, "B_cb": B_cb, "H_s": H_s, "H_h": H_h,
            "delta_Bs": delta_Bs, "delta_Hs": delta_Hs, "r": r,
            "alpha1": alpha1_t,
        }.items():
            getattr(self, name).append(float(value))

    def _print_table(self, indices, title, max_width=76):
        #Same plain style as Chapter 3; splits into blocks if periods overflow max_width.
        variables = [
            ("G", self.G), ("Y", self.Y), ("T", self.T), ("YD", self.YD),
            ("YD^e", self.YD_e), ("C", self.C), ("V", self.V),
            ("V^e", self.V_e), ("B^d", self.B_d), ("H^d", self.H_d),
            ("B_h", self.B_h), ("B_s", self.B_s), ("B_cb", self.B_cb),
            ("dB_s", self.delta_Bs), ("H_s", self.H_s), ("dH_s", self.delta_Hs),
            ("H_h", self.H_h), ("r", self.r), ("alpha1", self.alpha1),
        ]

        label_width, col_width = 10, 11

        indices = list(indices)
        cols_per_block = max(1, (max_width - label_width) // col_width)
        blocks = [indices[i:i + cols_per_block]
                  for i in range(0, len(indices), cols_per_block)]

        for block in blocks:
            header_line = f"{'Variable':<{label_width}}" + "".join(
                f"{'Period ' + str(self.period[i]):>{col_width}}" for i in block)
            total_width = len(header_line)

            block_title = title
            if len(blocks) > 1:
                block_title += (f"  (periods {self.period[block[0]]}"
                                 f"-{self.period[block[-1]]})")

            print("=" * total_width)
            print(block_title)
            print("=" * total_width)
            print(header_line)
            print("-" * total_width)

            for name, values in variables:
                row_line = f"{name:<{label_width}}" + "".join(
                    f"{values[i]:>{col_width}.2f}" for i in block)
                print(row_line)

            print("=" * total_width)
            print()

    def print_current_state(self):
        self._print_table([len(self.period) - 1], "CURRENT STATE \u2014 Model PCEX1")

    def print_history(self):
        self._print_table(range(len(self.period)), "SIMULATION RESULTS \u2014 Model PCEX1")

    def check_consistency(self, tol=1e-6):
        #Household wealth must equal bills supplied, and household money
        #holdings must equal money supplied, in every period.
        title = "STOCK-FLOW CONSISTENCY CHECK (V=B_s and H_h=H_s)"
        rows = []
        all_ok = True
        for i in range(len(self.period)):
            wealth_gap = self.V[i] - self.B_s[i]
            money_gap = self.H_h[i] - self.H_s[i]
            ok = max(abs(wealth_gap), abs(money_gap)) < tol
            all_ok = all_ok and ok
            symbol = "\u2713" if ok else "\u2717"
            rows.append(f"Period {self.period[i]:<3} "
                        f"V-B_s = {wealth_gap:>10.2e}   H_h-H_s = {money_gap:>10.2e}   "
                        f"[{symbol}]")
        summary = ("All periods consistent." if all_ok
                   else "Inconsistency detected -- check the equations.")

        inner = max([len(title), len(summary)] + [len(r) for r in rows]) + 2
        top = "\u256d" + "\u2500" * inner + "\u256e"
        mid = "\u251c" + "\u2500" * inner + "\u2524"
        bottom = "\u2570" + "\u2500" * inner + "\u256f"

        print()
        print(top)
        print("\u2502" + title.center(inner) + "\u2502")
        print(mid)
        for row in rows:
            print("\u2502 " + row.ljust(inner - 1) + "\u2502")
        print(mid)
        print("\u2502" + summary.center(inner) + "\u2502")
        print(bottom)


def run_economy(G, alpha1, alpha2, theta, r, YD_0, V_0, B_h_0,
                lambda20, lambda21, lambda22, interest_sensitivity=0.0):
    #Runs PCEX1 or PCEX2 using equally long lists for G, alpha1, alpha2, theta and r.
    series = [G, alpha1, alpha2, theta, r]
    if len({len(x) for x in series}) != 1:
        raise ValueError("G, alpha1, alpha2, theta and r must have the same length.")
    econ = PortfolioChoiceEconomy(YD_0, V_0, B_h_0, r_0=r[0])
    for values in zip(G, alpha1, alpha2, theta, r):
        econ.run_period(*values, lambda20, lambda21, lambda22, interest_sensitivity)
    econ.print_history()
    econ.check_consistency()
    return econ


econ = run_economy(
    G=[20, 20, 20],
    alpha1=[0.60, 0.60, 0.60],
    alpha2=[0.10, 0.10, 0.10],
    theta=[0.20, 0.20, 0.20],
    r=[0.03, 0.03, 0.03],
    YD_0=80,
    V_0=100,
    B_h_0=50,
    lambda20=0.65,
    lambda21=1.00,
    lambda22=0.10,
)

SIMULATION RESULTS — Model PCEX1
Variable     Period 0   Period 1   Period 2   Period 3
------------------------------------------------------
G                0.00      20.00      20.00      20.00
Y                0.00      78.00      68.72      65.40
T                0.00      15.90      14.18      13.53
YD              80.00      63.60      56.73      54.11
YD^e            80.00      80.00      63.60      56.73
C                0.00      58.00      48.72      45.40
V              100.00     105.60     113.61     122.32
V^e            100.00     122.00     120.48     124.94
B^d             50.00      73.20      74.67      78.72
H^d             50.00      48.80      45.81      46.22
B_h             50.00      73.20      74.67      78.72
B_s            100.00     105.60     113.61     122.32
B_cb            50.00      32.40      38.94      43.60
dB_s             0.00       5.60       8.01       8.71
H_s             50.00      32.40      38.94      43.60
dH_s             0.00     -17.60